# 🚀 GPU Notebooks: What Can You Do with Free GPUs?

Welcome! This notebook is a **speed tour** of what's possible when you have a GPU in the cloud — for free.

We'll cover:
1. ⚡ **GPU Speed Test** — How much faster is a GPU vs. CPU?
2. 🎨 **GPU Art** — Render a fractal in milliseconds
3. 🧠 **AI in One Line** — Image recognition with a pretrained model
4. ✨ **AI-Powered Coding** — Use Gemini to generate code from plain English

---

### ⚙️ First: Make sure you're on a GPU runtime!
`Runtime > Change runtime type > T4 GPU`

In [ ]:
# Quick GPU check — run this first!
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU is ready: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ No GPU detected — go to Runtime > Change runtime type > T4 GPU")

---

## ⚡ Demo 1: The GPU Speed Race

Let's multiply two massive matrices — first on the **CPU**, then on the **GPU** — and see who wins.

This is the fundamental operation behind all deep learning: matrix math at scale.

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

sizes = [1000, 3000, 5000, 8000]
cpu_times = []
gpu_times = []

print("🏁 Starting the race...\n")

for n in sizes:
    # --- CPU ---
    a_cpu = torch.randn(n, n)
    b_cpu = torch.randn(n, n)
    start = time.time()
    _ = torch.matmul(a_cpu, b_cpu)
    cpu_t = time.time() - start
    cpu_times.append(cpu_t)

    # --- GPU ---
    a_gpu = torch.randn(n, n, device='cuda')
    b_gpu = torch.randn(n, n, device='cuda')
    torch.cuda.synchronize()  # make sure GPU is ready
    start = time.time()
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()  # wait for GPU to finish
    gpu_t = time.time() - start
    gpu_times.append(gpu_t)

    speedup = cpu_t / gpu_t
    print(f"  {n}x{n} matrix multiply:  CPU {cpu_t:.3f}s  |  GPU {gpu_t:.4f}s  |  🚀 {speedup:.0f}x faster")

print(f"\n💡 At the largest size, the GPU was ~{cpu_times[-1]/gpu_times[-1]:.0f}x faster!")

In [ ]:
# Visualize the results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time comparison
x = np.arange(len(sizes))
width = 0.35
ax1.bar(x - width/2, cpu_times, width, label='CPU', color='#FF6B6B', alpha=0.9)
ax1.bar(x + width/2, gpu_times, width, label='GPU', color='#00C9A7', alpha=0.9)
ax1.set_xlabel('Matrix Size', fontsize=12)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('⏱️ CPU vs GPU: Time to Multiply', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{s}×{s}' for s in sizes])
ax1.legend(fontsize=12)
ax1.set_yscale('log')
ax1.grid(axis='y', alpha=0.3)

# Speedup chart
speedups = [c/g for c, g in zip(cpu_times, gpu_times)]
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(sizes)))
bars = ax2.bar(x, speedups, color=colors, alpha=0.9, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Matrix Size', fontsize=12)
ax2.set_ylabel('Speedup (×)', fontsize=12)
ax2.set_title('🚀 GPU Speedup Factor', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f'{s}×{s}' for s in sizes])
ax2.grid(axis='y', alpha=0.3)
for bar, s in zip(bars, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{s:.0f}×', ha='center', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

---

## 🎨 Demo 2: GPU Art — Mandelbrot Fractal

Let's use the GPU to render a **fractal** — an infinitely complex mathematical image. This would take a long time on a CPU, but the GPU handles millions of pixels in parallel.

In [ ]:
def mandelbrot_gpu(xmin, xmax, ymin, ymax, width=2000, height=2000, max_iter=200):
    """Render the Mandelbrot set entirely on the GPU using PyTorch."""
    # Create a grid of complex numbers on the GPU
    x = torch.linspace(xmin, xmax, width, device='cuda')
    y = torch.linspace(ymin, ymax, height, device='cuda')
    real, imag = torch.meshgrid(x, y, indexing='xy')
    c = torch.complex(real, imag)
    z = torch.zeros_like(c)
    result = torch.zeros(height, width, device='cuda')

    for i in range(max_iter):
        mask = z.abs() <= 2           # pixels that haven't escaped
        z[mask] = z[mask] ** 2 + c[mask]  # the fractal formula: z = z² + c
        result[mask] = i              # record when each pixel escapes

    return result.cpu().numpy()       # move back to CPU for plotting

# Render the full Mandelbrot set
print("🎨 Rendering 2000×2000 fractal on GPU...")
start = time.time()
fractal = mandelbrot_gpu(-2.0, 1.0, -1.5, 1.5)
elapsed = time.time() - start
print(f"✅ Done in {elapsed:.2f} seconds — that's {2000*2000:,} pixels!")

plt.figure(figsize=(12, 12))
plt.imshow(fractal.T, cmap='inferno', extent=[-2, 1, -1.5, 1.5])
plt.title('Mandelbrot Set — Rendered on GPU', fontsize=16, fontweight='bold')
plt.xlabel('Real')
plt.ylabel('Imaginary')
plt.colorbar(label='Iterations to escape', shrink=0.8)
plt.show()

In [ ]:
# 🔍 Zoom into a beautiful region — try changing these coordinates!
print("🔍 Zooming into the fractal...")
start = time.time()
zoom = mandelbrot_gpu(-0.75, -0.73, 0.1, 0.12, max_iter=500)
elapsed = time.time() - start
print(f"✅ Zoom rendered in {elapsed:.2f} seconds")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(fractal.T, cmap='inferno', extent=[-2, 1, -1.5, 1.5])
axes[0].add_patch(plt.Rectangle((-0.75, 0.1), 0.02, 0.02,
                                edgecolor='cyan', facecolor='none', linewidth=2))
axes[0].set_title('Full View', fontsize=14, fontweight='bold')

axes[1].imshow(zoom.T, cmap='magma', extent=[-0.75, -0.73, 0.1, 0.12])
axes[1].set_title('🔍 Zoomed In — Infinite Complexity!', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 🎯 Try It Yourself!
Change the coordinates in the zoom to explore different parts of the fractal. Some interesting spots:
- `(-0.16, -0.14, 1.03, 1.05)` — Seahorse Valley
- `(-1.26, -1.24, 0.04, 0.06)` — Mini Mandelbrot
- `(0.26, 0.28, 0.0, 0.02)` — Spiral Arms

---

## 🧠 Demo 3: AI Image Recognition

With one line of code, we can load a pretrained AI model that can recognize **thousands of objects** in images. This model was trained on millions of images — we're just using the result.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from torchvision import models, transforms

# Load a pretrained image classifier (ResNet-50, trained on ImageNet)
classifier = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).eval().cuda()
preprocess = models.ResNet50_Weights.DEFAULT.transforms()

# Load the ImageNet class labels
labels_url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
import json
labels = json.loads(requests.get(labels_url).text)

def classify_image(url, description=""):
    """Download an image and classify it with AI."""
    img = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0).cuda()

    with torch.no_grad():
        probs = torch.softmax(classifier(input_tensor), dim=1)
        top5 = torch.topk(probs, 5)

    # Display image and predictions side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                    gridspec_kw={'width_ratios': [1, 1.2]})
    ax1.imshow(img)
    ax1.axis('off')
    ax1.set_title(description or 'Input Image', fontsize=14, fontweight='bold')

    pred_labels = [labels[i] for i in top5.indices[0].cpu()]
    pred_probs = top5.values[0].cpu().numpy() * 100
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, 5))
    bars = ax2.barh(range(5), pred_probs[::-1], color=colors)
    ax2.set_yticks(range(5))
    ax2.set_yticklabels(pred_labels[::-1], fontsize=12)
    ax2.set_xlabel('Confidence (%)', fontsize=12)
    ax2.set_title('🤖 AI Predictions', fontsize=14, fontweight='bold')
    ax2.set_xlim(0, 105)
    for bar, pct in zip(bars, pred_probs[::-1]):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{pct:.1f}%', va='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✅ Model loaded! Let's classify some images...")

In [ ]:
# 🐕 Try a dog
classify_image(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg",
    "What kind of dog is this?"
)

# 🏔️ Try a landscape
classify_image(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/1280px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg",
    "What does the AI see here?"
)

In [ ]:
# 🎯 Try your own image! Paste any image URL below.
classify_image(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/300px-PNG_transparency_demonstration_1.png",
    "Your image"
)

---

## ✨ Demo 4: AI-Powered Coding with Gemini in Colab

Google Colab has a built-in AI coding assistant called **Gemini**. Instead of writing code yourself, you can describe what you want in plain English and Gemini will generate the code for you.

This is **AI-scaffolded development** — you bring the ideas, the AI helps you build them.

### 🔑 How to activate Gemini
1. Look for the **✨ sparkle icon** in the toolbar or in any empty code cell
2. You may need to **sign in** with your Google account and accept the terms
3. Once enabled, click ✨ in any code cell to type a prompt in plain English
4. Gemini will generate code — click **Insert** to add it to your cell, then run it!

### 🎮 Challenge 1: Make a Game!
In the empty cell below, click the **✨ Generate** button and ask Gemini to create something fun.

**Try this prompt:**
> `Create a simple number guessing game where the computer picks a random number between 1 and 100 and gives hints like 'too high' or 'too low'`

In [ ]:
# ✨ Use Gemini to generate a number guessing game!
# Click the sparkle icon ✨ or type your request as a comment and let Gemini write the code.


### 📊 Challenge 2: Create a Visualization
Ask Gemini to build something visual. The more specific your prompt, the cooler the result!

**Try one of these prompts:**
> `Create a colorful animated spiral using matplotlib that changes colors as it grows`

> `Plot a 3D surface that looks like ocean waves using numpy and matplotlib`

> `Generate a random colorful pixel art grid (16x16) and display it as an image`

In [ ]:
# ✨ Ask Gemini to create a cool visualization!


### 🎲 Challenge 3: Get Creative!
Gemini can help you build almost anything. Try one of these ideas — or make up your own!

| Idea | Prompt to Try |
|------|---------------|
| 🎲 Dice sim | `Simulate rolling 2 dice 10,000 times and plot the distribution of sums as a bar chart` |
| 🌍 Fun facts | `Write a function that prints a random fun science fact each time it's called, include at least 10 facts` |
| 🎨 ASCII art | `Create a function that turns any word into large ASCII block letters` |
| 📈 Stock sim | `Simulate a random stock price over 365 days using a random walk and plot it` |
| 🧬 DNA | `Generate a random DNA sequence of 100 bases, count each base, and plot the frequencies as a pie chart` |
| 🌀 Turtle art | `Use matplotlib to draw a colorful spirograph pattern with overlapping circles` |

**The key insight:** You don't need to know how to code all of this from scratch — you just need to know what you want to build. The AI handles the implementation.

In [ ]:
# ✨ Your turn — ask Gemini to build something awesome!


In [ ]:
# ✨ Bonus round — try another idea!


### 💡 Tips for getting great results from Gemini
- **Be specific** — *"Plot a bar chart of the top 10 most common words in a paragraph"* beats *"make a chart"*
- **Iterate** — If the first result isn't perfect, tweak your prompt or ask Gemini to modify the code
- **Ask it to explain** — Select any code and ask *"What does this do?"* to learn from what it generates
- **Build on previous cells** — *"Take the visualization above and add a title and legend"* works great
- **Don't be afraid to experiment** — The worst that happens is you get an error, and you can just try again!

---

## 🎓 What Did We Just Do?

| Demo | What You Saw | Why It Matters |
|------|-------------|----------------|
| ⚡ Speed Race | GPU was **50-100×** faster than CPU | Deep learning = millions of matrix multiplications |
| 🎨 Fractal Art | 4 million pixels rendered in seconds | GPU handles millions of calculations **in parallel** |
| 🧠 Image AI | Recognized objects instantly | Pretrained models run fast on GPU |
| ✨ Gemini Coding | AI wrote code from your plain English prompts | You bring the ideas, AI helps you build them |

### 🚀 Keep Exploring!
- **Google Colab**: [colab.research.google.com](https://colab.research.google.com) — free GPU notebooks with Gemini built in
- **Hugging Face**: [huggingface.co](https://huggingface.co) — thousands of pretrained AI models
- **PyTorch Tutorials**: [pytorch.org/tutorials](https://pytorch.org/tutorials)
- **Full Workshop Notebooks**: [github.com/emenriquez/MiniWorkshop_S25](https://github.com/emenriquez/MiniWorkshop_S25)